In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sharedfunctions import loadKernels, computeSigma, computeMagnitudeResponse, calculateEcc, eccL2Distance, eccWassersteinDistance, IDXROBUST, IDXVULNERABLE, GRIDSIZE, SIGMAGRID, NTHRESH, CSVPATH


In [6]:
def generateVpWaveletKernel():
    n = 3
    nodes = np.cos(np.pi * np.arange(1, 2*n, 2) / (2*n))
    vp2d = np.outer(nodes, nodes)
    vp2d = vp2d - vp2d.mean()
    vp2d = vp2d / np.max(np.abs(vp2d))
    return vp2d.astype(np.float32)

def fgsm_perturbation(h, epsilon):

    mag = computeMagnitudeResponse(h, grid=GRIDSIZE)
    flatidx = np.argsort(mag.ravel())[:20]
    rows, cols = np.unravel_index(flatidx, mag.shape)
    HPerturb = np.zeros((GRIDSIZE, GRIDSIZE), dtype=complex)
    for r, c in zip(rows, cols):
        HPerturb[r, c] = epsilon * GRIDSIZE**2
    perturb_full = np.real(np.fft.ifft2(np.fft.ifftshift(HPerturb)))
    center = GRIDSIZE // 2
    perturb_3x3 = perturb_full[center-1:center+2, center-1:center+2]
    return h + perturb_3x3

def computeAllL2(kernels, eccVp=None):
    """Compute L2 distance to VP Wavelet for all kernels."""
    if eccVp is None:
        vp = generateVpWaveletKernel()
        magVp = computeMagnitudeResponse(vp)
        _, eccVp = calculateEcc(magVp)
    all_l2 = []
    print("  Computing L2 distances for all kernels...")
    for i, h in enumerate(kernels):
        mag = computeMagnitudeResponse(h)
        _, ecc = calculateEcc(mag)
        all_l2.append(eccL2Distance(ecc, eccVp))
        if (i + 1) % 500 == 0:
            print(f"    {i+1}/{len(kernels)} done")
    return np.array(all_l2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sharedfunctions import (loadKernels, computeSigma, computeMagnitudeResponse,
                              calculateEcc, eccL2Distance, eccWassersteinDistance, generateVPWaveletKernel,
                              IDXROBUST, IDXVULNERABLE, GRIDSIZE, SIGMAGRID, NTHRESH, CSVPATH)

kernels = loadKernels(CSVPATH)

# ============================================================
# Compute sigmas and L2 ECC distances for all kernels
# ============================================================
print("Computing sigmas for all kernels...")
sigmas = np.array([computeSigma(h) for h in kernels])

vp = generateVPWaveletKernel()
magVp = computeMagnitudeResponse(vp)
_, eccVp = calculateEcc(magVp)

print("Computing L2 distances for all kernels...")
allL2 = []
for i, h in enumerate(kernels):
    mag = computeMagnitudeResponse(h)
    _, ecc = calculateEcc(mag)
    allL2.append(eccL2Distance(ecc, eccVp))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(kernels)} done")
allL2 = np.array(allL2)

# ============================================================
# Correlation Analysis
# ============================================================
print("\n" + "="*60)
print("Correlation Interpretation (sigma vs L2 ECC distance)")
print("="*60)

r = np.corrcoef(sigmas, allL2)[0, 1]
print(f"\n  Pearson r (sigma vs L2 ECC distance) = {r:.4f}")
print(f"\n  Interpretation:")
print(f"    sigma measures:    spectral distance of zero-variety from unit bi-circle")
print(f"    L2 ECC measures:   topological shape similarity to VP Wavelet")
print(f"    Weak r ({r:.3f}) is EXPECTED — they measure orthogonal properties.")
print(f"    A strong r would indicate redundancy between the two metrics.")
print(f"\n  Key anchors:")
print(f"    Kernel {IDXROBUST}  (Robust)    sigma={sigmas[IDXROBUST]:.4f}  L2={allL2[IDXROBUST]:.3f}")
print(f"    Kernel {IDXVULNERABLE} (Vulnerable) sigma={sigmas[IDXVULNERABLE]:.6e}  L2={allL2[IDXVULNERABLE]:.3f}")
print(f"    Best case:  high sigma AND low L2  → doubly robust  (Kernel {IDXROBUST})")
print(f"    Worst case: low sigma AND high L2  → doubly vulnerable (Kernel {IDXVULNERABLE})")

# Quadrant analysis split at medians
sigmaMed = np.median(sigmas)
l2Med    = np.median(allL2)
qRobust  = ((sigmas > sigmaMed) & (allL2 < l2Med)).sum()
qVuln    = ((sigmas < sigmaMed) & (allL2 > l2Med)).sum()
qMixed1  = ((sigmas > sigmaMed) & (allL2 > l2Med)).sum()
qMixed2  = ((sigmas < sigmaMed) & (allL2 < l2Med)).sum()

print(f"\n  Quadrant analysis (split at medians):")
print(f"    High sigma + Low L2  (doubly robust):     {qRobust:4d} kernels ({100*qRobust/len(sigmas):.1f}%)")
print(f"    Low sigma  + High L2 (doubly vulnerable): {qVuln:4d} kernels ({100*qVuln/len(sigmas):.1f}%)")
print(f"    High sigma + High L2 (sigma-safe only):   {qMixed1:4d} kernels ({100*qMixed1/len(sigmas):.1f}%)")
print(f"    Low sigma  + Low L2  (L2-safe only):      {qMixed2:4d} kernels ({100*qMixed2/len(sigmas):.1f}%)")

# ============================================================
# Figure 1: Correlation Scatter
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    "Sigma vs L2 ECC Distance — Complementary Metrics\n"
    f"Pearson r = {r:.3f}  |  Weak correlation is expected, not a flaw",
    fontsize=12, fontweight='bold'
)

# Left panel: scatter with trend line
ax = axes[0]
ax.scatter(sigmas, allL2, c='#5c6bc0', alpha=0.15, s=8, label='All kernels')
ax.scatter(sigmas[IDXROBUST], allL2[IDXROBUST],
           c='orange', s=120, zorder=5,
           label=f'Kernel {IDXROBUST} (Robust): σ={sigmas[IDXROBUST]:.4f}, L2={allL2[IDXROBUST]:.2f}')
ax.scatter(sigmas[IDXVULNERABLE], allL2[IDXVULNERABLE],
           c='blue', s=120, zorder=5,
           label=f'Kernel {IDXVULNERABLE} (Vulnerable): σ={sigmas[IDXVULNERABLE]:.2e}, L2={allL2[IDXVULNERABLE]:.2f}')

z = np.polyfit(sigmas, allL2, 1)
p = np.poly1d(z)
xLine = np.linspace(sigmas.min(), sigmas.max(), 100)
ax.plot(xLine, p(xLine), 'r--', linewidth=1.5, alpha=0.7, label=f'Trend (r={r:.3f})')

ax.axvline(sigmaMed, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
ax.axhline(l2Med,    color='gray', linewidth=0.8, linestyle=':', alpha=0.7)

ax.text(sigmas.max()*0.6, allL2.min()*1.1, 'Doubly\nRobust',
        fontsize=8, color='green', alpha=0.7, ha='center')
ax.text(sigmas.max()*0.02, allL2.max()*0.92, 'Doubly\nVulnerable',
        fontsize=8, color='red', alpha=0.7, ha='left')

ax.set_xlabel("Stability Margin (sigma)\n← Spectral distance from zero-variety to unit bi-circle",
              fontsize=10)
ax.set_ylabel("L2 Distance to VP Wavelet ECC\n← Topological shape similarity to gold standard",
              fontsize=10)
ax.set_title(f"Two Complementary Robustness Axes\nr = {r:.3f} confirms metrics are not redundant",
             fontsize=10, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right panel: quadrant coloring
ax2 = axes[1]
colorsq = np.full(len(sigmas), '#cccccc', dtype=object)
colorsq[(sigmas > sigmaMed) & (allL2 < l2Med)] = '#2ecc71'
colorsq[(sigmas < sigmaMed) & (allL2 > l2Med)] = '#e74c3c'
colorsq[(sigmas > sigmaMed) & (allL2 > l2Med)] = '#f39c12'
colorsq[(sigmas < sigmaMed) & (allL2 < l2Med)] = '#3498db'

ax2.scatter(sigmas, allL2, c=colorsq, alpha=0.3, s=8)
ax2.scatter(sigmas[IDXROBUST], allL2[IDXROBUST],
            c='green', s=150, zorder=5, edgecolors='black', linewidths=1.5,
            label=f'K{IDXROBUST}: doubly robust')
ax2.scatter(sigmas[IDXVULNERABLE], allL2[IDXVULNERABLE],
            c='red', s=150, zorder=5, edgecolors='black', linewidths=1.5,
            label=f'K{IDXVULNERABLE}: doubly vulnerable')

ax2.axvline(sigmaMed, color='gray', linewidth=0.8, linestyle=':')
ax2.axhline(l2Med,    color='gray', linewidth=0.8, linestyle=':')

legend_elements = [
    plt.scatter([], [], c='#2ecc71', s=40, label=f'Doubly Robust ({qRobust})'),
    plt.scatter([], [], c='#e74c3c', s=40, label=f'Doubly Vulnerable ({qVuln})'),
    plt.scatter([], [], c='#f39c12', s=40, label=f'Sigma-safe only ({qMixed1})'),
    plt.scatter([], [], c='#3498db', s=40, label=f'L2-safe only ({qMixed2})'),
]
ax2.legend(handles=legend_elements, fontsize=8, loc='upper right')
ax2.set_xlabel("Stability Margin (sigma)", fontsize=10)
ax2.set_ylabel("L2 Distance to VP Wavelet ECC", fontsize=10)
ax2.set_title("Why Both Metrics Are Needed\nNeither metric alone captures full robustness",
              fontsize=10, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("CorrelationScatter.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: CorrelationScatter.png")

# ============================================================
# Multi-Epsilon Sigma Degradation Table
# ============================================================
print("\n" + "="*60)
print("Multi-Epsilon Adversarial Sigma Degradation Table")
print("="*60)

epsilons = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5]

hRob          = kernels[IDXROBUST]
hVul          = kernels[IDXVULNERABLE]
sigmaRobClean = computeSigma(hRob)
sigmaVulClean = computeSigma(hVul)

print(f"\n  Clean baseline:")
print(f"    Kernel {IDXROBUST} (Robust):     sigma = {sigmaRobClean:.6f}")
print(f"    Kernel {IDXVULNERABLE} (Vulnerable): sigma = {sigmaVulClean:.2e}")

print(f"\n  {'epsilon':>8}  "
      f"{f'K{IDXROBUST} sigma':>14}  {f'K{IDXROBUST} drop%':>12}  "
      f"{f'K{IDXVULNERABLE} sigma':>14}  {f'K{IDXVULNERABLE} drop%':>12}")
print("  " + "-"*70)

resultsRob = []
resultsVul = []

for eps in epsilons:
    hRobAdv, _ = fsgmPerturbation(hRob, eps)
    hVulAdv, _ = fsgmPerturbation(hVul, eps)
    sr = computeSigma(hRobAdv)
    sv = computeSigma(hVulAdv)
    pctr = 100 * (sr - sigmaRobClean) / (sigmaRobClean + 1e-12)
    pctv = 100 * (sv - sigmaVulClean) / (sigmaVulClean + 1e-12)
    resultsRob.append(sr)
    resultsVul.append(sv)
    print(f"  {eps:>8.3f}  {sr:>14.6f}  {pctr:>+11.1f}%  {sv:>14.8f}  {pctv:>+11.1f}%")

print(f"\n  Interpretation:")
print(f"    Kernel {IDXROBUST} degrades gracefully — large margin absorbs perturbation.")
print(f"    Kernel {IDXVULNERABLE} has sigma≈0 at baseline — critically pre-failed.")

# ============================================================
# Figure 2: Epsilon Sweep
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Stability Margin Degradation — Full Epsilon Sweep\n"
    "Extends adversarial analysis from eps=0.05 to full threat model",
    fontsize=12, fontweight='bold'
)

# Left panel: absolute sigma vs epsilon
ax = axes[0]
ax.plot(epsilons, resultsRob, 'o-', color='orange', linewidth=2.5,
        markersize=8, label=f'Kernel {IDXROBUST} (Robust)  σ_clean={sigmaRobClean:.4f}')
ax.plot(epsilons, resultsVul, 's-', color='blue', linewidth=2.5,
        markersize=8, label=f'Kernel {IDXVULNERABLE} (Vulnerable)  σ_clean={sigmaVulClean:.2e}')

ax.axhspan(0,     0.001,              alpha=0.15, color='#ff6b6b', label='Critical (σ<0.001)')
ax.axhspan(0.001, 0.005,              alpha=0.15, color='#ffa94d', label='High Risk (0.001-0.005)')
ax.axhspan(0.005, 0.050,              alpha=0.10, color='#ffe066', label='Moderate (0.005-0.05)')
ax.axhspan(0.050, max(resultsRob)*1.1, alpha=0.10, color='#ccffcc', label='Robust (≥0.05)')
ax.axvline(0.05, color='gray', linewidth=1.5, linestyle='--', alpha=0.7, label='eps=0.05 reference')

ax.set_xlabel("Perturbation Magnitude (epsilon)", fontsize=10)
ax.set_ylabel("Stability Margin (sigma)", fontsize=10)
ax.set_title(f"Sigma vs Epsilon: Kernel {IDXROBUST} vs Kernel {IDXVULNERABLE}\n"
             "Robust kernel degrades gracefully; vulnerable is pre-failed",
             fontsize=10, fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=-0.01)

# Right panel: percentage retention
ax2 = axes[1]
pctRob = [100 * s / (sigmaRobClean + 1e-12) for s in resultsRob]

if sigmaVulClean > 1e-6:
    pctVul = [100 * s / sigmaVulClean for s in resultsVul]
else:
    pctVul = [0.0] * len(resultsVul)

ax2.plot(epsilons, pctRob, 'o-', color='orange', linewidth=2.5,
         markersize=8, label=f'Kernel {IDXROBUST} (Robust)')
ax2.plot(epsilons, pctVul, 's-', color='blue', linewidth=2.5,
         markersize=8, label=f'Kernel {IDXVULNERABLE} (Vulnerable)')
ax2.axhline(100, color='gray', linewidth=1,   linestyle=':',  alpha=0.5, label='Baseline (no attack)')
ax2.axhline(50,  color='red',  linewidth=1.2, linestyle='--', alpha=0.7, label='50% retention threshold')
ax2.axvline(0.05, color='gray', linewidth=1.5, linestyle='--', alpha=0.7, label='eps=0.05 reference')

ax2.set_xlabel("Perturbation Magnitude (epsilon)", fontsize=10)
ax2.set_ylabel("% of Clean Sigma Retained", fontsize=10)
ax2.set_title(f"Relative Sigma Retention Under Escalating Attack\n"
              f"Kernel {IDXVULNERABLE} — critically pre-failed at baseline",
              fontsize=10, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("EpsilonSweep.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: EpsilonSweep.png")

ImportError: cannot import name 'fsgmPerturbation' from 'sharedfunctions' (/Users/lilave/Desktop/Deterministic-Verification-of-CNN-Stability-via-the-Bivariate-Z-Transform/extracted_kernels/Notebooks/sharedfunctions.py)

In [7]:
kernels = loadKernels(CSVPATH)

In [8]:


print("\n" + "="*60)
print("Correlation Interpretation (sigma vs L2 ECC distance)")
print("Multi-Epsilon Adversarial Sigma Degradation Table")
print("="*60)

n     = 3
nodes = np.cos(np.pi * np.arange(1, 2*n, 2) / (2*n))
vp    = np.outer(nodes, nodes)
vp    = vp - vp.mean()
vp    = (vp / np.max(np.abs(vp))).astype(np.float32)


magVp    = computeMagnitudeResponse(vp)
_, eccVp = calculateEcc(magVp)

sigmas = np.array([computeSigma(h) for h in kernels])

print("Computing L2 distances for all kernels...")
allL2 = []
for i, h in enumerate(kernels):
    mag = computeMagnitudeResponse(h)
    _, ecc = calculateEcc(mag)
    allL2.append(eccL2Distance(ecc, eccVp))
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(kernels)} done")
allL2 = np.array(allL2)

    
r = np.corrcoef(sigmas, allL2)[0, 1]
print(f"\n  Pearson r (sigma vs L2 ECC distance) = {r:.4f}")
print(f"\n  Interpretation:")
print(f"    sigma measures:    spectral distance of zero-variety from unit bi-circle")
print(f"    L2 ECC measures:   topological shape similarity to VP Wavelet")
print(f"    Weak r ({r:.3f}) is EXPECTED — they measure orthogonal properties.")
print(f"    A strong r would indicate redundancy between the two metrics.")
print(f"\n  Key anchors:")
print(f"    Kernel {IDXROBUST}  (Robust)    sigma={sigmas[IDXROBUST]:.4f}  L2={allL2[IDXROBUST]:.3f}")
print(f"    Kernel {IDXVULNERABLE} (Vulnerable) sigma={sigmas[IDXVULNERABLE]:.6f}  L2={allL2[IDXVULNERABLE]:.3f}")
print(f"    Best case: high sigma AND low L2 → certified robust (Kernel {IDXROBUST})")
print(f"    Worst case: low sigma AND high L2 → doubly vulnerable")


sigmaMed = np.median(sigmas)
l2Med = np.median(allL2)
qRobust  = ((sigmas > sigmaMed) & (allL2 < l2Med)).sum()
qVuln    = ((sigmas < sigmaMed) & (allL2 > l2Med)).sum()
qMixed1  = ((sigmas > sigmaMed) & (allL2 > l2Med)).sum()
qMixed2  = ((sigmas < sigmaMed) & (allL2 < l2Med)).sum()
print(f"\n  Quadrant analysis (split at medians):")
print(f"    High sigma + Low L2  (doubly robust):     {qRobust:4d} kernels ({100*qRobust/len(sigmas):.1f}%)")
print(f"    Low sigma  + High L2 (doubly vulnerable): {qVuln:4d} kernels ({100*qVuln/len(sigmas):.1f}%)")
print(f"    High sigma + High L2 (spectrally safe):   {qMixed1:4d} kernels ({100*qMixed1/len(sigmas):.1f}%)")
print(f"    Low sigma  + Low L2  (topologically safe):{qMixed2:4d} kernels ({100*qMixed2/len(sigmas):.1f}%)")

 
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
        "Sigma vs L2 ECC Distance — Complementary Metrics\n"
        f"Pearson r = {r:.3f}  |  Weak correlation is expected, not a flaw",
        fontsize=12, fontweight='bold'
    )

ax = axes[0]
ax.scatter(sigmas, allL2, c='#5c6bc0', alpha=0.15, s=8, label='All kernels')
ax.scatter(sigmas[IDXROBUST], allL2[IDXROBUST],
               c='orange', s=120, zorder=5,
               label=f'Kernel {IDXROBUST} (Robust): σ={sigmas[IDXROBUST]:.4f}, L2={allL2[IDXROBUST]:.2f}')
ax.scatter(sigmas[IDXVULNERABLE], allL2[IDXVULNERABLE],
               c='blue', s=120, zorder=5,
               label=f'Kernel {IDXVULNERABLE} (Vulnerable): σ={sigmas[IDXVULNERABLE]:.2e}, L2={allL2[IDXVULNERABLE]:.2f}')

  
z = np.polyfit(sigmas, allL2, 1)
p = np.poly1d(z)
xLine = np.linspace(sigmas.min(), sigmas.max(), 100)
ax.plot(xLine, p(xLine), 'r--', linewidth=1.5, alpha=0.7,
            label=f'Trend (r={r:.3f})')


ax.axvline(sigmaMed, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
ax.axhline(l2Med, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)

    
ax.text(sigmas.max()*0.6, allL2.min()*1.1, 'Doubly\nRobust', fontsize=8,
            color='green', alpha=0.7, ha='center')
ax.text(sigmas.max()*0.02, allL2.max()*0.92, 'Doubly\nVulnerable', fontsize=8,
            color='red', alpha=0.7, ha='left')

ax.set_xlabel("Stability Margin (sigma)\n← Spectral distance from zero-variety to unit bi-circle", fontsize=10)
ax.set_ylabel("L2 Distance to VP Wavelet ECC\n← Topological shape similarity to gold standard", fontsize=10)
ax.set_title(f"Two Complementary Robustness Axes\nr = {r:.3f} confirms metrics are not redundant",
                 fontsize=10, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)


ax2 = axes[1]
   
colorsq = np.full(len(sigmas), '#cccccc')  
colorsq[(sigmas > sigmaMed) & (allL2 < l2Med)] = '#2ecc71'   
colorsq[(sigmas < sigmaMed) & (allL2 > l2Med)] = '#e74c3c'   
colorsq[(sigmas > sigmaMed) & (allL2 > l2Med)] = '#f39c12'   
colorsq[(sigmas < sigmaMed) & (allL2 < l2Med)] = '#3498db'   

ax2.scatter(sigmas, allL2, c=colorsq, alpha=0.3, s=8)
ax2.scatter(sigmas[IDXROBUST], allL2[IDXROBUST],
                c='green', s=150, zorder=5, edgecolors='black', linewidths=1.5,
                label=f'K{IDXROBUST}: doubly robust')
ax2.scatter(sigmas[IDXVULNERABLE], allL2[IDXVULNERABLE],
                c='red', s=150, zorder=5, edgecolors='black', linewidths=1.5,
                label=f'K{IDXVULNERABLE}: doubly vulnerable')

ax2.axvline(sigmaMed, color='gray', linewidth=0.8, linestyle=':')
ax2.axhline(l2Med, color='gray', linewidth=0.8, linestyle=':')

legend_elements = [
        plt.scatter([], [], c='#2ecc71', s=40, label=f'Doubly Robust ({qMixed1})'),
        plt.scatter([], [], c='#e74c3c', s=40, label=f'Doubly Vulnerable ({qMixed2})'),
        plt.scatter([], [], c='#f39c12', s=40, label=f'Sigma-safe only ({qMixed1})'),
        plt.scatter([], [], c='#3498db', s=40, label=f'L2-safe only ({qMixed2})'),
    ]
ax2.legend(handles=legend_elements, fontsize=8, loc='upper right')
ax2.set_xlabel("Stability Margin (sigma)", fontsize=10)
ax2.set_ylabel("L2 Distance to VP Wavelet ECC", fontsize=10)
ax2.set_title("Why Both Metrics Are Needed\nNeither metric alone captures full robustness",
                  fontsize=10, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("CorrelationScatter.png", dpi=150, bbox_inches='tight')
plt.close()


    
print("Multi-Epsilon Sigma Degradation Table")
print("="*60)

epsilons = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5]

print(f"\n  {'epsilon':>8}  {'K42 sigma':>12}  {'K42 drop%':>10}  "
          f"{'K806 sigma':>12}  {'K806 drop%':>10}")
print("  " + "-"*60)

resultsRob = []
resultsVul = []
hRob = kernels[IDXROBUST]
hVul = kernels[IDXVULNERABLE]
sigmaRobClean = computeSigma(hRob)
sigmaVulClean = computeSigma(hVul)

for eps in epsilons:
        sr = computeSigma(fgsm_perturbation(hRob, eps))
        sv = computeSigma(fgsm_perturbation(hVul, eps))
        pctr = 100 * (sr - sigmaRobClean) / (sigmaRobClean + 1e-12)
        pctv = 100 * (sv - sigmaVulClean) / (sigmaVulClean + 1e-12)
        resultsRob.append(sr)
        resultsVul.append(sv)
        print(f"  {eps:>8.3f}  {sr:>12.6f}  {pctr:>+9.1f}%  {sv:>12.8f}  {pctv:>+9.1f}%")

print(f"\n  Clean baseline:")
print(f"    Kernel {IDXROBUST} (Robust):     sigma = {sigmaRobClean:.6f}")
print(f"    Kernel {IDXVULNERABLE} (Vulnerable): sigma = {sigmaVulClean:.8f}")
print(f"\n  Interpretation:")
print(f"    Kernel {IDXVULNERABLE} is already at sigma≈0 before any perturbation.")
print(f"    Kernel {IDXROBUST} degrades gracefully — large margin absorbs noise.")

   
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
        "Stability Margin Degradation — Full Epsilon Sweep\n"
        "Extends Component 2 from eps=0.05 to full adversarial threat model",
        fontsize=12, fontweight='bold'
    )

   
ax = axes[0]
ax.plot(epsilons, resultsRob, 'o-', color='orange', linewidth=2.5,
            markersize=8, label=f'Kernel {IDXROBUST} (Robust)  σ_clean={sigmaRobClean:.4f}')
ax.plot(epsilons, resultsVul, 's-', color='blue', linewidth=2.5,
            markersize=8, label=f'Kernel {IDXVULNERABLE} (Vulnerable)  σ_clean={sigmaVulClean:.2e}')

    
ax.axhspan(0,     0.001, alpha=0.15, color='#ff6b6b', label='Critical (σ<0.001)')
ax.axhspan(0.001, 0.005, alpha=0.15, color='#ffa94d', label='High Risk (0.001-0.005)')
ax.axhspan(0.005, 0.050, alpha=0.10, color='#ffe066', label='Moderate (0.005-0.05)')
ax.axhspan(0.050, max(resultsRob)*1.1, alpha=0.10, color='#ccffcc', label='Robust (≥0.05)')

   
ax.axvline(0.05, color='gray', linewidth=1.5, linestyle='--', alpha=0.7,
               label='eps=0.05 (existing comp2 result)')

ax.set_xlabel("Perturbation Magnitude (epsilon)", fontsize=10)
ax.set_ylabel("Stability Margin (sigma)", fontsize=10)
ax.set_title("Sigma vs Epsilon for Robust vs Vulnerable Kernel\n"
                 "Robust kernel degrades gracefully; Vulnerable is pre-failed",
                 fontsize=10, fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=-0.01)


ax2 = axes[1]
pctRob = [100 * s / (sigmaRobClean + 1e-12) for s in resultsRob]

if sigmaVulClean > 1e-6:
    pctVul = [100 * s / sigmaVulClean for s in resultsVul]
else:
    pctVul = [0.0] * len(resultsVul)

ax2.plot(epsilons, pctRob, 'o-', color='orange', linewidth=2.5,
             markersize=8, label=f'Kernel {IDXROBUST} (Robust)')
ax2.plot(epsilons, pctVul, 's-', color='blue', linewidth=2.5,
             markersize=8, label=f'Kernel {IDXVULNERABLE} (Vulnerable) — pre-failed (σ≈0)')
ax2.axhline(100, color='gray', linewidth=1, linestyle=':', alpha=0.5, label='Baseline (no attack)')
ax2.axhline(50, color='red', linewidth=1.2, linestyle='--', alpha=0.7, label='50% retention threshold')
ax2.axvline(0.05, color='gray', linewidth=1.5, linestyle='--', alpha=0.7,
                label='eps=0.05 (comp2 reference)')

ax2.set_xlabel("Perturbation Magnitude (epsilon)", fontsize=10)
ax2.set_ylabel("% of Clean Sigma Retained", fontsize=10)
ax2.set_title("Relative Sigma Retention Under Escalating Attack\n"
              "K806 flat at 0% — already failed before any perturbation",
              fontsize=10, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("EpsilonSweep.png", dpi=150, bbox_inches='tight')
plt.close()





Correlation Interpretation (sigma vs L2 ECC distance)
Multi-Epsilon Adversarial Sigma Degradation Table
Computing L2 distances for all kernels...
  500/4096 done
  1000/4096 done
  1500/4096 done
  2000/4096 done
  2500/4096 done
  3000/4096 done
  3500/4096 done
  4000/4096 done

  Pearson r (sigma vs L2 ECC distance) = -0.2124

  Interpretation:
    sigma measures:    spectral distance of zero-variety from unit bi-circle
    L2 ECC measures:   topological shape similarity to VP Wavelet
    Weak r (-0.212) is EXPECTED — they measure orthogonal properties.
    A strong r would indicate redundancy between the two metrics.

  Key anchors:
    Kernel 42  (Robust)    sigma=0.8952  L2=24.576
    Kernel 806 (Vulnerable) sigma=0.000861  L2=33.675
    Best case: high sigma AND low L2 → certified robust (Kernel 42)
    Worst case: low sigma AND high L2 → doubly vulnerable

  Quadrant analysis (split at medians):
    High sigma + Low L2  (doubly robust):     1159 kernels (28.3%)
    Low sigma  